# 00. Synthetic Data Generator (E-Commerce Source)

Generates realistic synthetic datasets directly into the Unity Catalog landing volume:
`/Volumes/bronze_prod/ecommerce/raw_landing/`

### Features:
- **Zero external dependencies**: Built entirely using native PySpark functions.
- **CDC & SCD Type 2 Testing**: Parameterized via `batch_mode` widget (`initial` vs `update`).
- **Edge cases included**: Mixed casing, varied email domains, date ranges, order statuses.


In [0]:
%run ./01_Config


In [0]:
import random
from datetime import datetime, timedelta
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType
)
from pyspark.sql.functions import col, lit, current_timestamp

dbutils.widgets.dropdown("batch_mode", "initial", ["initial", "update"], "Batch Mode")
batch_mode = dbutils.widgets.get("batch_mode")
print(f"[DATA GENERATOR] Running in '{batch_mode}' mode")


### 1. Generate Region Reference Data


In [0]:
# Geographic Region Reference Data
region_data = [
    (1, "North America", "US", "USD", 0.0825),
    (2, "Western Europe", "DE", "EUR", 0.1900),
    (3, "United Kingdom", "GB", "GBP", 0.2000),
    (4, "Asia Pacific", "JP", "JPY", 0.1000),
    (5, "South America", "BR", "BRL", 0.1700),
    (6, "Oceania", "AU", "AUD", 0.1000),
    (7, "Middle East", "AE", "AED", 0.0500),
    (8, "Eastern Europe", "PL", "PLN", 0.2300)
]

region_schema = StructType([
    StructField("region_id", IntegerType(), False),
    StructField("region_name", StringType(), False),
    StructField("country_code", StringType(), False),
    StructField("currency", StringType(), False),
    StructField("tax_rate", DoubleType(), False)
])

df_region_raw = spark.createDataFrame(region_data, region_schema)
region_out_path = f"{RAW_LANDING_VOLUME}/region"
df_region_raw.write.mode("overwrite").parquet(region_out_path)
print(f"✓ Wrote {df_region_raw.count()} regions to {region_out_path}")


### 2. Generate Product Catalog Data


In [0]:
# Products Data Generator
categories = [
    ("Electronics", ["Smartphone", "Laptop Pro", "Wireless Earbuds", "4K Monitor", "Smart Watch", "Tablet Air", "Bluetooth Speaker"]),
    ("Apparel", ["Classic Denim Jacket", "Cotton T-Shirt", "Running Shoes", "Wool Sweater", "Athletic Shorts", "Leather Belt"]),
    ("Home & Kitchen", ["Espresso Machine", "Non-Stick Cookware Set", "Air Purifier", "Robot Vacuum", "Blender 1000W", "Chef Knife"]),
    ("Sports & Outdoors", ["Yoga Mat Pro", "Dumbbell Set 20kg", "Camping Tent 4P", "Mountain Bike", "Hydration Pack", "Fitness Tracker"]),
    ("Beauty & Personal", ["Hydrating Facial Cream", "Organic Shampoo", "Electric Toothbrush", "Sunscreen SPF50", "Perfume Eau de Parfum"])
]

random.seed(42)
products = []
pid = 1001

for cat_name, items in categories:
    for item in items:
        for variant in ["Standard", "Pro", "Elite", "Max"]:
            cost = round(random.uniform(10.0, 450.0), 2)
            margin = random.uniform(1.3, 2.2)
            price = round(cost * margin, 2)
            supplier = f"{cat_name.split()[0]} Global Direct"
            products.append((f"P{pid}", f"{item} {variant}", cat_name, price, cost, supplier))
            pid += 1

# If batch_mode is 'update', modify prices for 25 products to demonstrate SCD Type 2 updates!
if batch_mode == "update":
    print("[CDC] Simulating product price & attribute changes for SCD Type 2...")
    updated_products = []
    for p in products:
        if int(p[0][1:]) % 8 == 0:
            new_price = round(p[3] * 1.15, 2)  # 15% price increase
            updated_products.append((p[0], f"{p[1]} (Updated Edition)", p[2], new_price, p[4], p[5]))
        else:
            updated_products.append(p)
    products = updated_products

prod_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("cost_price", DoubleType(), False),
    StructField("supplier_name", StringType(), False)
])

df_products_raw = spark.createDataFrame(products, prod_schema)
products_out_path = f"{RAW_LANDING_VOLUME}/products"
df_products_raw.write.mode("overwrite").parquet(products_out_path)
print(f"✓ Wrote {df_products_raw.count()} products to {products_out_path}")


### 3. Generate Customer Data


In [0]:
# Customers Data Generator
first_names = ["James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael", "Linda", "David", "Elizabeth",
               "William", "Barbara", "Richard", "Susan", "Joseph", "Jessica", "Thomas", "Sarah", "Charles", "Karen"]
last_names = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez",
              "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "icloud.com", "corporate.net", "techcorp.io"]

random.seed(101)
customers = []
total_cust = 5000 if batch_mode == "initial" else 5200

for cid in range(10001, 10001 + total_cust):
    fn = random.choice(first_names)
    ln = random.choice(last_names)
    domain = random.choice(domains)
    email = f"{fn.lower()}.{ln.lower()}{cid % 1000}@{domain}"
    phone = f"+1-555-{random.randint(100, 999)}-{random.randint(1000, 9999)}"
    rid = random.randint(1, 8)
    reg_date = (datetime(2021, 1, 1) + timedelta(days=random.randint(0, 1095))).strftime("%Y-%m-%d")
    customers.append((f"C{cid}", fn, ln, email, phone, rid, reg_date))

# If batch_mode is 'update', modify email/phone for 150 customers to demonstrate SCD Type 2 updates!
if batch_mode == "update":
    print("[CDC] Simulating customer profile changes for SCD Type 2...")
    updated_customers = []
    for c in customers:
        cid_num = int(c[0][1:])
        if cid_num <= 10150:
            new_email = f"updated.{c[3]}"
            new_phone = "+1-555-999-0000"
            updated_customers.append((c[0], c[1], c[2], new_email, new_phone, c[5], c[6]))
        else:
            updated_customers.append(c)
    customers = updated_customers

cust_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("phone_number", StringType(), True),
    StructField("region_id", IntegerType(), False),
    StructField("registration_date", StringType(), False)
])

df_customers_raw = spark.createDataFrame(customers, cust_schema)
customers_out_path = f"{RAW_LANDING_VOLUME}/customers"
df_customers_raw.write.mode("overwrite").parquet(customers_out_path)
print(f"✓ Wrote {df_customers_raw.count()} customers to {customers_out_path}")


### 4. Generate Orders Transactions Data


In [0]:
# Orders Transactions Generator
random.seed(999)
statuses = ["COMPLETED", "COMPLETED", "COMPLETED", "SHIPPED", "PENDING", "CANCELLED", "RETURNED"]
prod_id_list = [p[0] for p in products]
prod_price_map = {p[0]: p[3] for p in products}
cust_id_list = [c[0] for c in customers[:5000]]

total_orders = 25000 if batch_mode == "initial" else 30000
start_date = datetime(2022, 1, 1)
orders = []

for oid in range(1, total_orders + 1):
    order_id = f"ORD-{oid:07d}"
    cid = random.choice(cust_id_list)
    pid = random.choice(prod_id_list)
    rid = random.randint(1, 8)
    qty = random.choices([1, 2, 3, 4, 5], weights=[0.55, 0.25, 0.12, 0.05, 0.03])[0]
    unit_p = prod_price_map.get(pid, 49.99)
    total_amt = round(qty * unit_p, 2)
    status = random.choice(statuses)
    days_offset = random.randint(0, 1095)
    seconds_offset = random.randint(0, 86399)
    order_ts = (start_date + timedelta(days=days_offset, seconds=seconds_offset)).strftime("%Y-%m-%d %H:%M:%S")
    orders.append((order_id, cid, pid, rid, qty, unit_p, total_amt, status, order_ts))

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("region_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("total_amount", DoubleType(), False),
    StructField("order_status", StringType(), False),
    StructField("order_date", StringType(), False)
])

df_orders_raw = spark.createDataFrame(orders, orders_schema)
orders_out_path = f"{RAW_LANDING_VOLUME}/orders"
df_orders_raw.write.mode("overwrite").parquet(orders_out_path)
print(f"✓ Wrote {df_orders_raw.count()} orders to {orders_out_path}")


### 5. Generation Summary


In [0]:
# Display verification row count summary
summary = [
    ("Region", df_region_raw.count(), region_out_path),
    ("Products", df_products_raw.count(), products_out_path),
    ("Customers", df_customers_raw.count(), customers_out_path),
    ("Orders", df_orders_raw.count(), orders_out_path)
]
df_summary = spark.createDataFrame(summary, ["Dataset", "RecordCount", "LandingPath"])
display(df_summary)
